## Cài đặt và Imports

In [ ]:
!pip install timm torchcam pandas scikit-learn seaborn grad-cam


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.utils.checkpoint import checkpoint

import torchvision.transforms as transforms
from PIL import Image
import pandas as pd
import numpy as np
import os
import timm
from tqdm.notebook import tqdm
import glob
import copy

# For visualization and interpretation
import torch.nn.utils.prune as prune
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
from torchcam.methods import GradCAM
from torchcam.utils import overlay_mask


## 1. Cấu hình Training

In [ ]:
CONFIG = {
    # Model Selection: Choose from timm library
    # Examples: 'resnet50', 'efficientnet_b0' to 'efficientnet_b7', 'vit_base_patch16_224',
    # 'swin_base_patch4_window7_224', 'regnety_040', 'densenet121', 'inception_v3'
    'model_name': 'efficientnet_b0',
    'num_classes': 2,  # benign, malignant
    'image_size': 224,
    'batch_size': 32,
    'num_workers': 4,
    'learning_rate': 1e-4,
    'epochs': 50,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42,
    
    # Paths
    'data_csv_path': '/path/to/isic2024/train-metadata.csv', # Path to ISIC 2024 metadata
    'image_folder_path': '/path/to/isic2024/train-image/', # Path to ISIC 2024 images
    'checkpoint_path': './checkpoints',
    'results_path': './results',

    # Early Stopping
    'early_stopping_patience': 5,
    'max_checkpoints': 3, # Keep top 3 best models

    # Metadata features
    # Đây là danh sách các cột sẽ được sử dụng LÀM ĐẶC TRƯNG SAU KHI TIỀN XỬ LÝ.
    # Nó sẽ được tự động tạo ra trong bước xử lý dữ liệu.
    # Bạn không cần chỉnh sửa danh sách này thủ công.
    'metadata_features': [], 
    'metadata_embedding_dim': 64, # Size of metadata embedding

    # Advanced Optimizations
    'use_gradient_checkpointing': True, # Use gradient checkpointing to save memory
    'pruning_amount': 0.3, # Prune 30% of weights post-training
}

# Create directories
os.makedirs(CONFIG['checkpoint_path'], exist_ok=True)
os.makedirs(CONFIG['results_path'], exist_ok=True)

# Set seed for reproducibility
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
if CONFIG['device'] == 'cuda':
    torch.cuda.manual_seed(CONFIG['seed'])


## 2. Dataset và DataLoader đa phương thức

In [ ]:
class MultimodalDataset(Dataset):
    def __init__(self, df, image_folder, metadata_features, image_transform=None, is_train=True):
        self.df = df
        self.image_folder = image_folder
        self.metadata_features = metadata_features
        self.image_transform = image_transform
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row['isic_id'] # Changed to use 'isic_id'
        image_path = os.path.join(self.image_folder, f"{image_id}.jpg")
        image = Image.open(image_path).convert('RGB')

        if self.image_transform:
            image = self.image_transform(image)

        metadata = torch.tensor(row[self.metadata_features].values.astype(np.float32))
        
        if self.is_train:
            label = torch.tensor(row['label'], dtype=torch.long)
            return image, metadata, label
        else:
            return image, metadata

# Data Augmentation and Normalization
train_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- Hướng dẫn xử lý dữ liệu thật của ISIC 2024 ---
# 1. Bỏ comment các dòng dưới đây khi bạn sử dụng dữ liệu thật.
# 2. Đảm bảo cột chứa nhãn (target) và các cột metadata khác có tên chính xác.
from sklearn.model_selection import train_test_split

full_df = pd.read_csv(CONFIG['data_csv_path'], low_memory=False)

# In ra danh sách các cột để kiểm tra tên cột nhãn chính xác
print("Các cột có trong tệp CSV:", full_df.columns.tolist())

# Ánh xạ nhãn từ text sang số
# Sử dụng cột 'benign_malignant' như trong danh sách bạn cung cấp
label_mapping = {'benign': 0, 'malignant': 1}
full_df['label'] = full_df['benign_malignant'].map(label_mapping)

# Xử lý lỗi: Loại bỏ các dòng có nhãn không hợp lệ (NaN) sau khi map
initial_rows = len(full_df)
full_df.dropna(subset=['label'], inplace=True)
final_rows = len(full_df)
if initial_rows > final_rows:
    print(f"Cảnh báo: Đã loại bỏ {initial_rows - final_rows} dòng do có nhãn không hợp lệ trong cột 'benign_malignant'.")
full_df['label'] = full_df['label'].astype(int) # Chuyển cột label sang kiểu integer

# Lấy một mẫu 20,000 dòng từ dữ liệu gốc để thử nghiệm
# Sử dụng stratified sampling để giữ nguyên tỉ lệ các lớp
if len(full_df) > 20000:
    _, df = train_test_split(full_df, test_size=20000, random_state=CONFIG['seed'], stratify=full_df['label'])
    print(f"Đã lấy mẫu 20,000 dòng từ {len(full_df)} dòng dữ liệu gốc.")
else:
    df = full_df
    print(f"Sử dụng toàn bộ {len(df)} dòng dữ liệu.")

# Tạo một bản sao rõ ràng để tránh SettingWithCopyWarning
df = df.copy()

# --- TIỀN XỬ LÝ DỮ LIỆU METADATA MỞ RỘNG ---

# 1. Chọn các cột số và cột phân loại để làm đặc trưng
numerical_cols = ['age_approx', 'clin_size_long_diam_mm', 'mel_mitotic_index', 'mel_thick_mm']
categorical_cols = ['anatom_site_general', 'concomitant_biopsy', 'dermoscopic_type', 
                    'diagnosis_confirm_type', 'family_hx_mm', 'fitzpatrick_skin_type', 
                    'image_type', 'mel_ulcer', 'nevus_type', 'personal_hx_mm', 'sex']

# 2. Xử lý giá trị thiếu (Imputation)
# Đối với cột số: điền bằng giá trị trung bình
for col in numerical_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce') # Đảm bảo là kiểu số
        df[col].fillna(df[col].mean(), inplace=True)

# Đối với cột phân loại: điền bằng một giá trị riêng 'unknown'
for col in categorical_cols:
    if col in df.columns:
        df[col].fillna('unknown', inplace=True)

# 3. Chuẩn hóa các cột số (Min-Max Scaling)
for col in numerical_cols:
    if col in df.columns:
        df[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

# 4. One-Hot Encoding cho các cột phân loại
df = pd.get_dummies(df, columns=[col for col in categorical_cols if col in df.columns], dummy_na=False)

# 5. Cập nhật danh sách đặc trưng cuối cùng vào CONFIG một cách tường minh
# Lấy tên các cột số đã được xử lý
final_numerical_cols = [col for col in numerical_cols if col in df.columns]
# Lấy tên các cột mới được tạo ra từ one-hot encoding
one_hot_cols = [col for col in df.columns if any(cat_col in col for cat_col in categorical_cols)]

CONFIG['metadata_features'] = final_numerical_cols + one_hot_cols
print(f"Sử dụng {len(CONFIG['metadata_features'])} đặc trưng metadata.")

# Phân chia tập train/validation từ 20,000 mẫu đã chọn
train_df, val_df = train_test_split(df, test_size=0.2, random_state=CONFIG['seed'], stratify=df['label'])

train_dataset = MultimodalDataset(train_df, CONFIG['image_folder_path'], CONFIG['metadata_features'], train_transform)
val_dataset = MultimodalDataset(val_df, CONFIG['image_folder_path'], CONFIG['metadata_features'], val_transform)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)


## 3. Định nghĩa mô hình đa phương thức

In [ ]:
class MultimodalModel(nn.Module):
    def __init__(self, model_name, num_classes, metadata_features_count, metadata_embedding_dim, pretrained=True, use_grad_checkpoint=False):
        super().__init__()
        self.use_grad_checkpoint = use_grad_checkpoint
        # Image Encoder
        self.image_encoder = timm.create_model(model_name, pretrained=pretrained, num_classes=0) # num_classes=0 to get features
        num_image_features = self.image_encoder.num_features

        # Metadata Encoder
        self.metadata_encoder = nn.Sequential(
            nn.Linear(metadata_features_count, metadata_embedding_dim * 2),
            nn.BatchNorm1d(metadata_embedding_dim * 2),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(metadata_embedding_dim * 2, metadata_embedding_dim),
            nn.BatchNorm1d(metadata_embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )

        # Classifier
        self.classifier = nn.Linear(num_image_features + metadata_embedding_dim, num_classes)

    def forward(self, image, metadata):
        if self.use_grad_checkpoint and self.training:
            image_features = checkpoint(self.image_encoder, image, use_reentrant=False)
        else:
            image_features = self.image_encoder(image)
        metadata_features = self.metadata_encoder(metadata)
        
        combined_features = torch.cat([image_features, metadata_features], dim=1)
        
        output = self.classifier(combined_features)
        return output

model = MultimodalModel(
    model_name=CONFIG['model_name'],
    num_classes=CONFIG['num_classes'],
    metadata_features_count=len(CONFIG['metadata_features']),
    metadata_embedding_dim=CONFIG['metadata_embedding_dim'],
    use_grad_checkpoint=CONFIG['use_gradient_checkpointing']
).to(CONFIG['device'])

print(f"Model {CONFIG['model_name']} loaded on {CONFIG['device']}.")


## 4. Loss Function, Optimizer và Early Stopping

In [ ]:
# Focal Loss for imbalanced datasets
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

criterion = FocalLoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
scaler = GradScaler() # For Automatic Mixed Precision

class EarlyStopping:
    def __init__(self, patience=5, verbose=False, delta=0, path_prefix='checkpoint', max_checkpoints=3):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.delta = delta
        self.path_prefix = path_prefix
        self.max_checkpoints = max_checkpoints
        self.checkpoints = [] # List of (score, path)

    def __call__(self, val_loss, model, epoch):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model, epoch)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model, epoch)
            self.counter = 0

    def save_checkpoint(self, val_loss, model, epoch):
        path = f"{self.path_prefix}_epoch{epoch+1}_loss{val_loss:.4f}.pt"
        if self.verbose:
            print(f'Validation loss improved. Saving model to {path}')
        torch.save(model.state_dict(), path)
        self.checkpoints.append((-val_loss, path))
        self.checkpoints.sort(key=lambda x: x[0], reverse=True) # Best scores first

        if len(self.checkpoints) > self.max_checkpoints:
            worst_checkpoint = self.checkpoints.pop()
            worst_path = worst_checkpoint[1]
            if self.verbose:
                print(f"Removing old checkpoint: {worst_path}")
            if os.path.exists(worst_path):
                os.remove(worst_path)

    def get_best_model_path(self):
        if not self.checkpoints:
            return None
        return self.checkpoints[0][1]

checkpoint_prefix = os.path.join(CONFIG['checkpoint_path'], f"{CONFIG['model_name']}")
early_stopper = EarlyStopping(patience=CONFIG['early_stopping_patience'], verbose=True, path_prefix=checkpoint_prefix, max_checkpoints=CONFIG['max_checkpoints'])


## 5. Vòng lặp Training và Validation

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(CONFIG['epochs']):
    # --- Training Step ---
    model.train()
    running_loss = 0.0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']} [Train]")
    
    for images, metadata, labels in train_pbar:
        images = images.to(CONFIG['device'])
        metadata = metadata.to(CONFIG['device'])
        labels = labels.to(CONFIG['device'])

        optimizer.zero_grad()

        # Mixed Precision
        with autocast():
            outputs = model(images, metadata)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        train_pbar.set_postfix({'loss': loss.item()})

    epoch_train_loss = running_loss / len(train_loader.dataset)
    history['train_loss'].append(epoch_train_loss)

    # --- Validation Step ---
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']} [Val]")

    with torch.no_grad():
        for images, metadata, labels in val_pbar:
            images = images.to(CONFIG['device'])
            metadata = metadata.to(CONFIG['device'])
            labels = labels.to(CONFIG['device'])

            outputs = model(images, metadata)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            val_pbar.set_postfix({'loss': loss.item()})

    epoch_val_loss = val_loss / len(val_loader.dataset)
    epoch_val_acc = correct / total
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)

    print(f"Epoch {epoch+1}: Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")

    # Early stopping
    early_stopper(epoch_val_loss, model, epoch)
    if early_stopper.early_stop:
        print("Early stopping triggered")
        break

# Load best model
best_model_path = early_stopper.get_best_model_path()
if best_model_path:
    model.load_state_dict(torch.load(best_model_path))
    print(f"Loaded best model weights from {best_model_path} for evaluation.")
else:
    print("No checkpoint was saved. Using the last state of the model for evaluation.")


## 6. Trực quan hóa và Diễn giải

### 6.1. Grad-CAM

In [ ]:
def generate_grad_cam(model, img_tensor, target_layer):
    """Generate and display Grad-CAM for a single image."""
    model.eval()
    cam_extractor = GradCAM(model, target_layer)
    
    # Pre-process the image
    input_tensor = img_tensor.unsqueeze(0).to(CONFIG['device'])
    
    # Dummy metadata for Grad-CAM (it's not used for the CAM itself but required by forward pass)
    dummy_metadata = torch.zeros(1, len(CONFIG['metadata_features'])).to(CONFIG['device'])
    
    # Get model output
    scores = model(input_tensor, dummy_metadata)
    class_idx = scores.squeeze(0).argmax().item()
    
    # Generate CAM
    activation_map = cam_extractor(class_idx, scores)
    
    # Overlay on original image
    result = overlay_mask(transforms.ToPILImage()(img_tensor), transforms.ToPILImage()(activation_map[0].squeeze(0), mode='F'), alpha=0.5)
    
    plt.imshow(result)
    plt.axis('off')
    plt.title(f'Grad-CAM for class {class_idx}')
    plt.show()

# Find the target layer for Grad-CAM
# This depends on the model architecture. Common choices:
# - ResNet: model.image_encoder.layer4
# - EfficientNet: model.image_encoder.conv_head
# - ViT: model.image_encoder.blocks[-1].norm1
try:
    target_layer = model.image_encoder.layer4 # For ResNet-like models
except AttributeError:
    try:
        target_layer = model.image_encoder.conv_head # For EfficientNet
    except AttributeError:
        target_layer = model.image_encoder.blocks[-1].norm1 # For ViT/Swin
        print("Using last norm layer for ViT/Swin as target.")

# Get a sample image from validation set
sample_img, _, _ = val_dataset[0]
generate_grad_cam(model, sample_img, target_layer)


### 6.2. t-SNE

In [ ]:
def plot_tsne(model, data_loader, device):
    """Extract features and plot t-SNE."""
    model.eval()
    all_features = []
    all_labels = []

    # Hook to extract features from before the final classifier
    features = {}
    def get_features(name):
        def hook(model, input, output):
            features[name] = output.detach()
        return hook
    
    # Register hook on the input to the final classifier layer
    model.classifier.register_forward_hook(get_features('feats'))

    with torch.no_grad():
        for images, metadata, labels in tqdm(data_loader, desc="Extracting features for t-SNE"):
            images = images.to(device)
            metadata = metadata.to(device)
            
            _ = model(images, metadata)
            
            all_features.append(features['feats'].cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_features = np.concatenate(all_features)
    all_labels = np.concatenate(all_labels)

    print("Running t-SNE...")
    tsne = TSNE(n_components=2, random_state=CONFIG['seed'], perplexity=30, n_iter=1000)
    tsne_results = tsne.fit_transform(all_features)

    df_tsne = pd.DataFrame(tsne_results, columns=['tsne1', 'tsne2'])
    df_tsne['label'] = all_labels

    plt.figure(figsize=(12, 8))
    sns.scatterplot(
        x="tsne1", y="tsne2",
        hue="label",
        palette=sns.color_palette("hsv", CONFIG['num_classes']),
        data=df_tsne,
        legend="full",
        alpha=0.7
    )
    plt.title('t-SNE visualization of learned features')
    plt.savefig(os.path.join(CONFIG['results_path'], 'tsne_visualization.png'))
    plt.show()

plot_tsne(model, val_loader, CONFIG['device'])


### 6.3. Model Pruning

In [ ]:
def print_model_size(model, label=""):
    """Prints the size of the model's state_dict."""
    torch.save(model.state_dict(), "temp.p")
    size_mb = os.path.getsize("temp.p")/1e6
    print(f"Model size ({label}): {size_mb:.2f} MB")
    os.remove('temp.p')

def prune_model(model, amount=0.3):
    """Prunes the model globally and makes the pruning permanent."""
    print("\n--- Starting Model Pruning ---")
    print_model_size(model, "Original")

    # Identify parameters to prune (Linear and Conv layers)
    parameters_to_prune = []
    for module_name, module in model.named_modules():
        if isinstance(module, (torch.nn.Linear, torch.nn.Conv2d)):
            parameters_to_prune.append((module, 'weight'))

    if not parameters_to_prune:
        print("No prunable layers found.")
        return model

    # Apply global unstructured pruning
    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount,
    )

    # Make the pruning permanent by removing the re-parameterization
    for module, param_name in parameters_to_prune:
        prune.remove(module, param_name)

    print(f"Pruning complete. {amount*100:.0f}% of weights removed.")
    print_model_size(model, "Pruned")
    return model

# Prune the loaded best model
pruned_model = prune_model(model, amount=CONFIG['pruning_amount'])

# You can now save the pruned model
pruned_model_path = os.path.join(CONFIG['checkpoint_path'], f"{CONFIG['model_name']}_pruned.pt")
torch.save(pruned_model.state_dict(), pruned_model_path)
print(f"Pruned model saved to {pruned_model_path}")
